# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa

**Actividad:** construir el notebook `02_fundamentos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), aplicando extracción, transformaciones, funciones, agrupaciones/agregaciones, RDD y verificando en cada paso el efecto de la evaluación perezosa — sobre el dataset real H&M (Kaggle).

**Propósito de la actividad:** dejar evidencia ejecutable de que dominas el ciclo completo de transformación distribuida en PySpark — DataFrame y RDD sobre la misma `SparkSession` — antes de avanzar a formatos analíticos particionados (S3) y ML distribuido (S4).

Guía completa: `docs/sesiones/S02_Fundamentos_PySpark_Transformaciones_Lazy_Evaluation.md`, sección 3 (pasos 3.1 a 3.11).

## 3.1 Descargar el dataset H&M y reanudar el entorno `lambda26`

**Producto del paso:** dataset H&M disponible en `pyspark/sesiones/s02-fundamentos/data/`, entorno `lambda26` funcionando.

El dataset ya está descargado en `data/` (`articles.csv`, `customers.csv`, `transactions.parquet`). Si el contenedor `lambda26` sigue corriendo desde S1, continúa directo en 3.2.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

Declara la ruta del dataset como variable global, una sola vez — úsala en cada lectura del resto del notebook.

In [ ]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"
ARTIFACTS = "/opt/s02-fundamentos/artifacts"

## 3.3 Cargar y explorar `articles.csv`

**Producto del paso:** DataFrame `df_articles` cargado (primera forma de lectura: `inferSchema`), con estructura, filas y estadísticas verificadas paso a paso.

Primero, la lectura:

In [ ]:
df_articles = spark.read.csv(
    f"{ORIGEN_DATOS}/articles.csv",
    header=True,
    inferSchema=True,
)

`.show()` — visualiza filas en formato tabla:

In [ ]:
df_articles.show(5, truncate=False)

`.printSchema()` — muestra el esquema (nombres, tipos, nulabilidad):

In [ ]:
df_articles.printSchema()

`.describe()` — resumen estadístico. Con las 25 columnas de `articles.csv`, ni siquiera `vertical=True` lo deja cómodo (125 líneas) — mejor selecciona antes un puñado de columnas representativas, mezclando nominales y numéricas:

In [ ]:
df_articles.select(
    "prod_name", "product_group_name", "colour_group_name", "department_no", "section_no"
).describe().show()

Parámetros de `.show()`: `vertical=True` muestra cada fila como lista de campos, útil con muchas columnas.

In [ ]:
df_articles.show(3, vertical=True)

## 3.4 Cargar `customers.csv` con esquema explícito y explorar columnas

**Producto del paso:** DataFrame `df_customers` cargado con `StructType` (segunda forma de lectura), con sus columnas y su tamaño verificados.

Define el esquema, columna por columna:

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

Lee el CSV con ese esquema:

In [ ]:
df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

Resumen estadístico:

In [ ]:
df_customers.describe().show()

Nombres de columna:

In [ ]:
print(df_customers.columns)

Cantidad de filas y de columnas:

In [ ]:
num_rows, num_cols = df_customers.count(), len(df_customers.columns)
print(f"Filas: {num_rows}, Columnas: {num_cols}")

**Muestra aleatoria** (`.sample()`): con ~1.37 millones de filas, trabajar con todo el dataset en una laptop se vuelve pesado — para explorar y validar la lógica alcanza con una muestra.

In [ ]:
df_customers_muestra = df_customers.sample(
    withReplacement=False,  # sin reemplazo: cada fila se elige como máximo una vez
    fraction=0.1,           # ~10% del dataset
    seed=None,              # sin semilla fija: cada corrida da una muestra distinta
)

**Registros iniciales** (`.head()`):

In [ ]:
df_customers_muestra.head(3)

**Registros finales** (`.tail()`):

In [ ]:
df_customers_muestra.tail(3)

**Guardar la muestra en CSV y Parquet:** adelanto de lo que S3 formaliza a fondo — por ahora, solo la mecánica básica de escribir un resultado a disco. Spark escribe una **carpeta** (un archivo por partición adentro), no un solo archivo; `mode("overwrite")` reemplaza la carpeta si ya existe.

In [ ]:
df_customers_muestra.write.mode("overwrite").csv(f"{ARTIFACTS}/customers_muestra_csv", header=True)
df_customers_muestra.write.mode("overwrite").parquet(f"{ARTIFACTS}/customers_muestra_parquet")

## 3.5 Aplicar transformaciones y verificar la evaluación perezosa

**Producto del paso:** evidencia de que el plan se construye antes de ejecutarse.

Primero, `.select()` — elige columnas específicas del DataFrame:

In [ ]:
df_seleccionado = df_customers.select("customer_id", "age", "club_member_status", "fashion_news_frequency")

Ahora, `.filter()` — conserva solo las filas que cumplen una condición:

In [ ]:
from pyspark.sql.functions import col

df_activos = df_seleccionado.filter(col("club_member_status") == "ACTIVE")

Como práctica, combina ambas en una sola expresión encadenada:

In [ ]:
df_activos = (
    df_customers
    .select("customer_id", "age", "club_member_status", "fashion_news_frequency")
    .filter(col("club_member_status") == "ACTIVE")
)

# Hasta aquí Spark solo construyó el plan: no hay salida, no hubo ejecución
df_activos

Acción: aquí recién Spark ejecuta.

In [ ]:
df_activos.show(10, truncate=False)
df_activos.count()

## 3.6 Analizar el plan de ejecución con `explain()`

**Producto del paso:** plan de ejecución interpretado con al menos una optimización identificada.

In [ ]:
df_activos.explain(True)

## 3.7 Aplicar funciones y crear columnas con `withColumn()`

**Producto del paso:** `df_articles` (cargado en 3.3) con al menos tres columnas nuevas o corregidas.

In [ ]:
from pyspark.sql.functions import col, when, lit, current_date

**Corregir el tipo de una columna** (`.cast()`): `article_id` son solo dígitos, `inferSchema` corre el riesgo de quitarle el cero inicial.

In [ ]:
df_articles = df_articles.withColumn("article_id", col("article_id").cast("string"))

**Clasificar con `when()`/`otherwise()`** (valores reales de `perceived_colour_value_name`: `"Dark"`, `"Light"`, u otros):

In [ ]:
df_articles = df_articles.withColumn(
    "rango_percibido",
    when(col("perceived_colour_value_name") == "Dark", "oscuro")
    .when(col("perceived_colour_value_name") == "Light", "claro")
    .otherwise("medio")
)

**Agregar una columna constante** (`lit()`):

In [ ]:
df_articles = df_articles.withColumn("fuente", lit("H&M Kaggle"))

**Agregar una columna con fecha actual** (`current_date()` — la calcula Spark, no tú):

In [ ]:
df_articles = df_articles.withColumn("fecha_procesado", current_date())

Verifica el resultado — confirma que `article_id` conserva el cero inicial (ej. `0108775015`, no `108775015`):

In [ ]:
df_articles.select(
    "article_id", "prod_name", "perceived_colour_value_name", "rango_percibido", "fuente", "fecha_procesado"
).show(5, truncate=False)

## 3.8 Cargar `transactions.parquet` y aplicar funciones

**Producto del paso:** DataFrame `df_transactions` cargado (tercera forma de lectura: Parquet), con tipos corregidos y columnas clasificadas para poder agrupar en 3.9.

Primero, la lectura — trabajamos solo con un subconjunto (`.limit(100000)`) del archivo completo, para explorar sin procesar todo el volumen disponible:

In [ ]:
df_transactions = spark.read.parquet(f"{ORIGEN_DATOS}/transactions.parquet").limit(100000)

Confirma tú mismo las columnas — no asumas el esquema:

In [ ]:
df_transactions.printSchema()

Corrige los tipos con `withColumn()` encadenado, sobre las 5 columnas reales:

In [ ]:
from pyspark.sql.functions import col, to_date

df_transactions = (
    df_transactions
    .withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("article_id", col("article_id").cast("string"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("sales_channel_id", col("sales_channel_id").cast("int"))
)

**Clasificar por canal de venta:**

In [ ]:
from pyspark.sql.functions import when

df_transactions = df_transactions.withColumn(
    "canal",
    when(col("sales_channel_id") == 1, "Online")
    .when(col("sales_channel_id") == 2, "Tienda")
    .otherwise("Desconocido")
)

**Etiquetar transacciones baratas o caras:** recuerda que `price` está normalizado a [0, 1] — los umbrales van en esa escala, no en soles/dólares:

In [ ]:
df_transactions = df_transactions.withColumn(
    "categoria_precio",
    when(col("price") < 0.1, "Barato")
    .when(col("price") < 0.3, "Medio")
    .otherwise("Caro")
)

**Aplicar múltiples condiciones** (con `&`, combinando dos columnas):

In [ ]:
df_transactions = df_transactions.withColumn(
    "tipo_transaccion",
    when((col("sales_channel_id") == 1) & (col("price") > 0.3), "Online Premium")
    .when((col("sales_channel_id") == 2) & (col("price") > 0.3), "Tienda Premium")
    .otherwise("Regular")
)

**Fin de semana vs. laborable** (`date_format()` + `.isin()` — `"u"` es el día ISO de la semana, 1=lunes...7=domingo, no confundir con `"d"` que es día del mes):

In [ ]:
from pyspark.sql.functions import date_format

df_transactions = df_transactions.withColumn("dia_semana", date_format(col("t_dat"), "u"))

df_transactions = df_transactions.withColumn(
    "tipo_dia",
    when(col("dia_semana").isin("6", "7"), "Fin de semana").otherwise("Laborable")
)

## 3.9 Aplicar agrupaciones y agregaciones (`transactions.parquet`)

**Producto del paso:** resumen agregado por cliente, con la advertencia de dominio sobre `price` aplicada.

**Total gastado por cliente** (`sum`):

In [ ]:
from pyspark.sql.functions import sum

df_total_por_cliente = df_transactions.groupBy("customer_id").agg(
    sum("price").alias("total_normalizado")
)
df_total_por_cliente.show(5)

**Promedio de gasto por cliente** (`avg`):

In [ ]:
from pyspark.sql.functions import avg

df_avg_por_cliente = df_transactions.groupBy("customer_id").agg(
    avg("price").alias("promedio_normalizado")
)
df_avg_por_cliente.show(5)

**Número de transacciones por cliente** (`count`):

In [ ]:
from pyspark.sql.functions import count

df_count_por_cliente = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones")
)
df_count_por_cliente.show(5)

**Clientes únicos por día** (`countDistinct`):

In [ ]:
from pyspark.sql.functions import countDistinct

df_clientes_unicos_por_dia = df_transactions.groupBy("t_dat").agg(
    countDistinct("customer_id").alias("clientes_unicos")
)
df_clientes_unicos_por_dia.show(5)

**Varias agregaciones a la vez** (más eficiente que calcularlas por separado):

In [ ]:
from pyspark.sql.functions import sum, count, avg

df_agg_multi = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones"),
    sum("price").alias("total_normalizado"),
    avg("price").alias("promedio_normalizado")
)
df_agg_multi.show(5)

**Agrupar por múltiples columnas:**

In [ ]:
df_ventas_por_dia_y_canal = df_transactions.groupBy("t_dat", "sales_channel_id").agg(
    sum("price").alias("total_normalizado")
)
df_ventas_por_dia_y_canal.show(5)

**Usar `agg()` con un diccionario** (forma alternativa, sin `alias()`):

In [ ]:
df_agg_dict = (
    df_transactions.groupBy("customer_id")
    .agg({"price": "sum", "article_id": "count"})
    .withColumnRenamed("sum(price)", "total_normalizado")
    .withColumnRenamed("count(article_id)", "num_articulos")
)
df_agg_dict.show(5)

**Total gastado por cliente, sin agrupar** (función ventana — a diferencia de `groupBy().agg()`, no colapsa filas):

In [ ]:
from pyspark.sql.window import Window

window_cliente = Window.partitionBy("customer_id")

In [ ]:
from pyspark.sql.functions import sum

df_con_total_cliente = df_transactions.withColumn(
    "total_gastado_cliente",
    sum("price").over(window_cliente)
)
df_con_total_cliente.show(5)

**Advertencia de dominio:** `price` está normalizado por Kaggle a [0, 1] — no representa una moneda real. `sum("price")`/`avg("price")` son agregaciones técnicamente correctas, pero leerlas como "gasto en soles/dólares" sería un error de dominio.

## 3.10 Convertir a RDD y procesar texto (`detail_desc` de `articles.csv`)

**Producto del paso:** conteo de palabras distribuido sobre descripciones de producto, con las 10 más frecuentes.

In [ ]:
import re
from operator import add

rdd = df_articles.select("detail_desc").rdd.map(lambda x: x.detail_desc)
rdd = rdd.filter(lambda texto: texto is not None)  # algunos artículos no tienen descripción

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)

pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

## 3.11 Documentar hallazgos y responder preguntas de reflexión

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

Agrega debajo de cada bloque anterior una breve explicación de qué hiciste y qué observaste — es la base directa de la evidencia técnica para 4.3.1.

**Reflexión técnica breve** (5 a 8 líneas): ¿por qué la evaluación perezosa es útil para procesar datos a escala, y qué riesgo tendría si Spark ejecutara cada transformación de inmediato, apenas se escribe?

_(Responde aquí)_